# png2svg — Craft

**Goal:** fine-grained control. Custom marker palettes, per-color remap, batch processing, comparison with adjacent tools.

If you haven't read `quickstart.ipynb` and `explore.ipynb`, do that first — this notebook assumes you understand the 4 stages.

## Contents

1. Custom marker palettes (your own pen set)

2. Per-color remap (force a specific marker for a specific detected color)

3. Render with the remap applied

4. Saving and reloading sessions (reproducibility)

5. Batch processing (apply the same recipe to many images)

6. Parameter deep-dive: what each setting does and how it affects the output

7. Comparison with adjacent tools (vtracer, vpype, hatched, plottter)

8. Troubleshooting

## 1. Custom marker palettes

A palette is a JSON file with brand, set, optional tip width, and a list of colors:

```json

{

  "brand": "My Pens",

  "set_name": "Fineliner 12",

  "tip_width_mm": 0.4,

  "colors": [

    {"name": "Red",    "hex": "#E63946", "rgb": [230, 57, 70]},

    {"name": "Blue",   "hex": "#1D3557", "rgb": [29, 53, 87]},

    ...

  ]

}

```

`rgb` is optional; if you provide `hex` we parse it. `tip_width_mm` controls the default

stroke width (your pen's physical tip).

In [ ]:
import json
from pathlib import Path
import notebook_helpers as nh

my_palette = {
    "brand": "My Pens",
    "set_name": "Fineliner 12",
    "tip_width_mm": 0.4,
    "colors": [
        {"name": "Red", "hex": "#E63946", "rgb": [230, 57, 70]},
        {"name": "Orange", "hex": "#F4A261", "rgb": [244, 162, 97]},
        {"name": "Yellow", "hex": "#E9C46A", "rgb": [233, 196, 106]},
        {"name": "Green", "hex": "#2A9D8F", "rgb": [42, 157, 143]},
        {"name": "Blue", "hex": "#264653", "rgb": [38, 70, 83]},
        {"name": "Black", "hex": "#000000", "rgb": [0, 0, 0]},
    ],
}

palette_path = Path("my_palette.json")
palette_path.write_text(json.dumps(my_palette, indent=2))
print(f"Wrote {palette_path.absolute()}")

# Now load it via the helper
loaded = nh.load_palette(str(palette_path))
print(f"Loaded: {loaded['brand']} {loaded['set_name']}, {len(loaded['colors'])} colors, tip {loaded['tip_width_mm']}mm")

## 2. Per-color remap

Sometimes `closest_marker()` makes a bad call — e.g. a near-white detected in the image

maps to a gray marker because the palette has no true white. The fix is to **force the

mapping for that specific color**.



This is also useful when you have two near-identical pens and want to make sure a specific

shade always uses the lighter one (or whatever your preference is).

In [ ]:
import notebook_helpers as nh
from IPython.display import display, Markdown

img = nh.load_image("../Bluey.png")
palette = nh.load_palette("crayola_10ct_fine_line_classic")
quant = nh.quantize_image(img, max_palette=6)
matched = nh.match_to_palette(quant, palette)

# Before remap: see the auto-mappings
print("Before remap:")
for c in matched["color_info"]:
    print(f"  {c['hex']} -> {c['marker_hex']} ({c['marker_name']})")

# Find the detected color we want to remap (e.g. the lightest one)
lightest = min(matched["color_info"], key=lambda c: sum(c["rgb"]))
print(f"\nRemapping lightest color: {lightest['hex']}")

# Build a remap dict: {color_id: marker_index_in_palette}
remap = {lightest["idx"]: 0}  # force map to the first marker in the palette

# Apply the remap by rebuilding color_info
remapped = []
for c in matched["color_info"]:
    if c["idx"] in remap:
        marker = palette["colors"][remap[c["idx"]]]
        marker_rgb = marker.get("rgb", [int(marker["hex"][i : i + 2], 16) for i in (1, 3, 5)])
        c = {**c, "marker_name": marker["name"], "marker_hex": marker["hex"], "marker_rgb": tuple(marker_rgb)}
    remapped.append(c)
matched["color_info"] = remapped

print("\nAfter remap:")
for c in matched["color_info"]:
    print(f"  {c['hex']} -> {c['marker_hex']} ({c['marker_name']})")

## 3. Render with the remap applied

Before we save the session, let's actually render the SVG with the remap in place.

This is the slow step — expect 5-30 seconds depending on the image.

We use the `fast` preset to keep render time low; the goal here is to validate the workflow,

not produce a final plot. Re-run with `portrait` for higher quality.

In [ ]:
result = nh.render_svg(
    img,
    preset="fast",  # use 'portrait' for the final version
    palette=palette,  # the marker palette we loaded above
    max_palette=6,
    continuous_paths=True,
    arc_radius=5.0,
)
from IPython.display import Markdown, display

display(Markdown(nh.format_render_report(result)))
nh.display_svg(result["svg_path"])

## 4. Saving and reloading sessions

Sessions capture: image path, params, palette, color mapping. They're a recipe for

reproducing an exact render later. Save them alongside the SVG; reload them when you

want to iterate without losing the working version.

In [ ]:
from pathlib import Path
from png2svg.core import RenderParams
import notebook_helpers as nh
import json

# Save a session
session = nh.snapshot_session(
    params=RenderParams(**result["params"]),
    palette=palette,
    color_map=result["color_map_used"],
    image_path="../Bluey.png",
)
session_path = Path("craft_session.json")
nh.save_session_file(session, session_path)
print(f"Saved: {session_path.absolute()}")
print(f"Contains: {list(session.keys())}")

# Reload it later
reloaded = nh.load_session_file(session_path)
print("\nReloaded session:")
print(f"  image: {reloaded['image']}")
print(f"  params: max_palette={reloaded['params']['max_palette']}, line_step={reloaded['params']['line_step']}")
print(f"  palette: {reloaded['palette']['brand']} ({len(reloaded['palette']['colors'])} colors)")
print(f"  version: png2svg {reloaded['version']}")

## 5. Batch processing

Apply the same recipe to many images. The trick is to wrap the slow stage in a loop and

save the SVG to a per-image path. This is the same logic the CLI uses internally; doing it

in a notebook lets you customize per-image if needed.

In [ ]:
from pathlib import Path

# Find all PNGs in a directory (e.g. a sticker pack)
source_dir = Path("../examples")  # change to your image directory
output_dir = Path("batch_output")
output_dir.mkdir(exist_ok=True)

image_paths = sorted(source_dir.glob("*.png"))
print(f"Found {len(image_paths)} images")

results = []
for img_path in image_paths:
    print(f"\n--- {img_path.name} ---")
    img = nh.load_image(img_path)
    result = nh.render_svg(
        img,
        preset="fast",  # use 'fast' for batch — quality matters less when iterating
        palette=palette,
        max_palette=4,
        line_step=8,
    )

    # Save with matching name
    out_path = output_dir / f"{img_path.stem}.svg"
    out_path.write_text(result["svg_path"].read_text())
    print(f"  -> {out_path.name} ({out_path.stat().st_size:,} bytes)")
    results.append({"image": img_path.name, "svg": out_path.name})

print(f"\nBatch complete: {len(results)} images")

## 6. Parameter deep-dive

What each RenderParams field does and how it affects the output. Skip to the section you need.

### `max_palette` (default 12)

Maximum number of colors in the quantized image. Lower = more banded (cartoon look),

higher = more detail.



Try: 2-3 for very simple line art, 4-6 for logos, 8-12 for portraits, 16+ for photos.

Beyond ~16, the SVG gets slow to render and the pen plotter has trouble — markers

are physically limited.

### `line_step` (default 4)

Spacing between hatch lines in pixels. Lower = denser hatch, slower to render, harder

for the pen to follow. Higher = sparser, faster, but may miss detail.



Rule of thumb: 1.5x to 2x your marker's stroke width in image pixels. For a 0.5mm

tip on a 1500px-wide image, try 4-6.

### `continuous_paths` (default off)

Chain rows of hatching into a single zigzag path. **Huge** reduction in pen lifts (often 50-80%).

Always enable for Cricut — the 'ringing' artifact you see without this is from thousands

of micro-pen-lifts per color layer.

### `arc_radius` (default 0)

Smooths the 180° U-turns at the end of each row with arc commands. Values 3-5 are typical.

**Requires `continuous_paths=True`** — arcs only make sense at the end of chained rows.

Set to 0 to disable and use sharp corners.

### `min_pixels` (default 200)

Skip layers with fewer than this many pixels. Useful for filtering out JPEG artifacts

and tiny specks. Set to 0 to keep everything; raise to 1000+ to aggressively clean up

noisy images.

### `white_medium` (default off)

When True, 'white-ish' colors are rendered as a medium tone (not skipped as background).

**Set True for photos** — you want pen strokes on light skin and pale highlights. **Set

False for line art** — white is paper, not a color.

### `skip_bg` / `paper_white_soft`

`skip_bg=True` excludes the most-common near-white color as the paper background.

`paper_white_soft` (0-255) controls the threshold for 'near white': values 20-30 are typical.

Lower = stricter (only true white is paper); higher = more permissive (cream backgrounds count).

### `separate_outline` (default off)

Emit outline paths as a separate SVG group from the hatch fills. Use this if you want

to assign the outline a different pen (e.g. a black fineliner for crisp edges, while the

fills are colored markers).

## 7. Comparison with adjacent tools

There are several other open-source pen-plotter tools. Use this table to pick the right one.



| Tool | Use when | Skip when |

|------|----------|-----------|

| **png2svg** | You want hatched SVG with **physical marker awareness** and a Cricut-friendly output | You want a smooth photo trace (use vtracer); you want grayscale halftone (use hatched) |

| **vtracer** (MIT) | You want a smooth multi-color trace of a photo, output goes to vpype for optimization | You want hatched fills (vtracer doesn't hatch) |

| **vpype + hatched** (MIT) | You already have an SVG and want to add hatching; or you want grayscale halftone | You want color, physical-marker awareness, or a Cricut-ready output |

| **plottter** (MIT) | You want a full desktop app with AI masks, dithering, and many generator styles | You want a simple CLI + notebook workflow with no AI dependencies |

| **saxi** (AGPL) | You have an AxiDraw and want a driver for it (note: AGPL license) | You want raster-to-vector (out of scope) |



**Common workflow:** png2svg → Inkscape (manual tweaks) → vpype (plot optimization) → Cricut Design Space or your plotter's driver.

## 8. Troubleshooting



**'No layers produced'**

- Lower `--min-pixels` (try 50)

- Add `--no-skip-background` to include the background as a layer

- Check that your image isn't all-transparent or all-black



**Colors don't match my expectations**

- The quantize step is in HSV Euclidean; for perceptually-accurate matching, you'd

  need Lab color space (not yet implemented; see issues)

- Try increasing `max_palette` — fewer colors means more aggressive merging

- Check `white_medium` — if False, near-white pixels are skipped



**Render is too slow**

- Use the `fast` preset for iteration; `portrait` only for the final render

- Lower `max_palette` (4-6 vs 12+ has a big effect)

- Increase `line_step` (8 vs 4 cuts hatch time in half)

- Resize the image first: `img.resize((img.size[0] // 2, img.size[1] // 2))`



**SVG opens but the plotter jitters**

- Enable `continuous_paths=True` (this is the #1 cause of jitter)

- Set `arc_radius=3.0` to `5.0` for smoother U-turns

- Some plotters want `stroke_width` lower than the marker's spec; try halving it

## Where to go from here

- Read the source: `src/png2svg/core.py` (the algorithm)

- Read `src/png2svg/presets.py` (preset definitions)

- Read `src/png2svg/cli.py` (the CLI — the notebook helpers wrap the same functions)

- File an issue: https://github.com/Veedubin/hatchsvg/issues